In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_sa
db_sql = "ODIN"
user_sql = user_sa
pwd_sql = pwd_sa
engine_sa = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

def nombre_mes_anio(fecha_mes_base):
    from datetime import datetime

    fecha = datetime.strptime(fecha_mes_base, "%Y-%m-%d")

    meses = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]

    return f"{meses[fecha.month - 1]} {fecha.year}"


In [2]:
filename='TARGET_EFECTIVO_202607.xlsx'

ruta_archivo = os.path.join(ruta_ventas_desembolso, filename)
df_desembolso_cli = pd.read_excel(ruta_archivo,sheet_name='Base')

df_desembolso_cli["FECHA_HORA_DESEMBOLSO"] = pd.to_datetime(df_desembolso_cli["FECHA_HORA_DESEMBOLSO"], errors="coerce")

fecha_min = df_desembolso_cli["FECHA_HORA_DESEMBOLSO"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_desembolso_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)


from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.efectiva_ventas_desembolso
        WHERE FECHA_HORA_DESEMBOLSO >= '{fecha_desembolso}'
          AND FECHA_HORA_DESEMBOLSO <= EOMONTH('{fecha_desembolso}');
    """))

df_desembolso_cli.to_sql(
    name="efectiva_ventas_desembolso",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)





2026-07-01 00:00:00


41

In [3]:
filename='Target_Efectinegocio_202607.xlsx'

ruta_archivo = os.path.join(ruta_ventas_desembolso, filename)
df_desembolso_cli = pd.read_excel(ruta_archivo,sheet_name='Base')

df_desembolso_cli["FECHA_DESEMBOLSO"] = pd.to_datetime(df_desembolso_cli["FECHA_DESEMBOLSO"], errors="coerce")

fecha_min = df_desembolso_cli["FECHA_DESEMBOLSO"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_desembolso_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)


from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.efectiva_negocios_ventas_desembolso
        WHERE FECHA_DESEMBOLSO >= '{fecha_desembolso}'
          AND FECHA_DESEMBOLSO <= EOMONTH('{fecha_desembolso}');
    """))

df_desembolso_cli.to_sql(
    name="efectiva_negocios_ventas_desembolso",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)





2026-07-01 00:00:00


67

In [4]:
# DECLARE @Hora VARCHAR(5) = FORMAT(GETDATE(), 'HH:mm');



# DECLARE @RutaBase VARCHAR(300) = 'I:\SQLServer Compartido\6.Desembolso\CencosudTC\SFTP_CENCO_SCOTIA\';



# DECLARE @Archivo1 VARCHAR(500) = NULL; -- Habilitadas

# DECLARE @Archivo2 VARCHAR(500) = NULL; -- Emboce



# DECLARE @Adjuntos VARCHAR(1000) = '';

# DECLARE @ListaArchivos NVARCHAR(MAX) = '';



# -- =========================================

# -- 🔹 BUSCAR HABILITADAS

# -- =========================================

# CREATE TABLE #H (linea VARCHAR(500));



# INSERT INTO #H

# EXEC xp_cmdshell 'dir /b /o-d "I:\SQLServer Compartido\6.Desembolso\CencosudTC\SFTP_CENCO_SCOTIA\Habilitada*.xlsx"';



# SELECT TOP 1 @Archivo1 = @RutaBase + linea

# FROM #H 

# WHERE linea IS NOT NULL

# AND linea NOT LIKE '%No se encuentra%'

# AND linea NOT LIKE '%File Not Found%'

# AND linea LIKE '%.xlsx';



# -- =========================================

# -- 🔹 BUSCAR EMBOCE

# -- =========================================

# CREATE TABLE #E (linea VARCHAR(500));



# INSERT INTO #E

# EXEC xp_cmdshell 'dir /b /o-d "I:\SQLServer Compartido\6.Desembolso\CencosudTC\SFTP_CENCO_SCOTIA\Emboce*.xlsx"';



# SELECT TOP 1 @Archivo2 = @RutaBase + linea

# FROM #E 

# WHERE linea IS NOT NULL

# AND linea NOT LIKE '%No se encuentra%'

# AND linea NOT LIKE '%File Not Found%'

# AND linea LIKE '%.xlsx';



# -- =========================================

# -- 🔹 ARMAR ADJUNTOS

# -- =========================================

# IF @Archivo1 IS NOT NULL

# BEGIN

#     SET @Adjuntos = @Archivo1;

#     SET @ListaArchivos = @ListaArchivos + REPLACE(@Archivo1, @RutaBase, '') + CHAR(13)+CHAR(10);

# END



# IF @Archivo2 IS NOT NULL

# BEGIN

#     SET @Adjuntos = 

#         CASE 

#             WHEN @Adjuntos = '' THEN @Archivo2

#             ELSE @Adjuntos + ';' + @Archivo2

#         END;



#     SET @ListaArchivos = @ListaArchivos + REPLACE(@Archivo2, @RutaBase, '') + CHAR(13)+CHAR(10);

# END



# -- =========================================

# -- 🔹 SI NO HAY ARCHIVOS → NO ENVÍA

# -- =========================================

# IF @Adjuntos = ''

# BEGIN

#     PRINT 'No hay archivos para enviar hoy';

#     DROP TABLE #H;

#     DROP TABLE #E;

#     RETURN;

# END



# -- =========================================

# -- 🔹 BODY DINÁMICO

# -- =========================================

# DECLARE @BodyMail NVARCHAR(MAX);



# SET @BodyMail =

#     'Hola equipo,' + CHAR(13)+CHAR(10)+CHAR(13)+CHAR(10) +

#     'Se adjuntan los archivos disponibles del SFTP (CENCOSUD):' + CHAR(13)+CHAR(10) +

#     @ListaArchivos + CHAR(13)+CHAR(10) +

#     'Hora: ' + @Hora + CHAR(13)+CHAR(10)+CHAR(13)+CHAR(10) +

#     'Data.';



# -- =========================================

# -- 🔹 ENVÍO DE CORREO

# -- =========================================

# EXEC msdb.dbo.sp_send_dbmail

#     @profile_name = 'TargetMensajes',

#     @recipients   = 'analista.datos@targetoutsourcing.com.pe;

# 					 rosa.novoa@targetoutsourcing.com.pe;

# 					 supervisor.cencosud1@targetoutsourcing.com.pe;

# 					 avillegh@gmail.com',

#     @subject      = 'CENCOSUD | Emboce - Habilitadas TARGET',

#     @body         = @BodyMail,

#     @body_format  = 'TEXT',

#     @file_attachments = @Adjuntos;



# -- =========================================

# -- 🔹 LIMPIEZA

# -- =========================================

# DROP TABLE #H;

# DROP TABLE #E;